In [3]:
import random
import pandas as panda

# Two assumptions :
- any damage string is called an occurrence, whether it's successful or not
- to separate occurrences, we add the letter A to the string to indicate which ones are modified by a trigger check

In [ ]:
def is_attack(occ):
    if isinstance(occ, str) and occ.upper().endswith('A'):
        return int(occ[:-1]), True
    return int(occ), False

In [4]:
def simulate_process(taille_deck, nombre_climax, occurrences_parsed, trigger_soul, trigger_blank):

    # Main deck definition, with its discard pile
    deck = ['Climax'] * taille_deck + ['Non-Climax'] * (taille_deck - nombre_climax)
    random.shuffle(deck)
    discard = []

    # Trigger deck definition, with its discard pile
    trigger_deck = ['Soul trigger'] * trigger_soul + ['No soul trigger'] * trigger_blank
    random.shuffle(trigger_deck)
    trigger_discard = []

    # Clock zone and total damage initialisation
    clock_zone = []
    total_kept = 0
 
    for base_k, is_A in occurrences_parsed:
        k = base_k
 
        # Trigger check here
        if is_A:
            if not trigger_deck and trigger_discard:
                trigger_deck = trigger_discard[:]
                trigger_discard = []
                random.shuffle(trigger_deck)
            if trigger_deck:
                card = trigger_deck.pop()
                trigger_discard.append(card)
                if card == 'Soul trigger':
                    k += 1
 
        # Main occurrence system
        drawn_this_occurrence = []
        occurrence_failed = False
        halted = False
        draws_done = 0
 
        while draws_done < k:
            if len(deck) == 0:
                # Here goes the refresh penalty system
                # Each time we go through the whole main system, we take one sure damage to ensure the game can end
                if not discard:
                    halted = True
                    break
                new_deck = discard[:]
                discard = []
                random.shuffle(new_deck)
                clock_card = new_deck.pop()  
                # for the sake of simplication, we assume here the card sent to level zone is sent at random
                                clock_zone.append(clock_card)
                total_kept += 1 
                deck = new_deck

                # in case the main deck size reaches 1 or 0, the processus is instantly killed
                # this matches the losing condition in the game
                if len(deck) == 1:
                    halted = True
                    break
                continue
 
            card = deck.pop()
            drawn_this_occurrence.append(card)
            draws_done += 1
            if card == 'Climax':
                occurrence_failed = True
                break
 
        if halted:
            discard.extend(drawn_this_occurrence)
            break
 
        if occurrence_failed:
            discard.extend(drawn_this_occurrence)
        else:
            # in case no climax is revealed, we update the damage count accordingly and send every revealed card to clock zone
            total_kept += len(drawn_this_occurrence)
            clock_zone.extend(drawn_this_occurrence)
 
            # level-up sytem
            while len(clock_zone) >= 7:
                batch = clock_zone[:7]
                clock_zone = clock_zone[7:]
                white_idx = next((i for i, c in enumerate(batch) if c == 'Non-Climax'), None)
                if white_idx is not None:
                    batch.pop(white_idx)
                else:
                    batch.pop(0)
                discard.extend(batch)
 
    return total_kept

In [2]:
def simulation(N_list, p_list, occurrence_list, trigger_soul, trigger_blank, n_trials=100000):
    """
    The simulator performs Monte Carlo simulations according to the parameters and creates the corresponding DataFrame
 
    Paramètres
    ----------
    N_list : list[int]        main deck size sample list
    p_list : list[int]        climax count sample list
    occurrences : list        damage string to test
    trigger_soul : int        soul trigger counts in trigger deck, we assume there's no double soul triggers for the moment
    trigger_blank : int       non trigger count in trigger deck
    n_trials : int            Monte Carlo simulations par [N,p] combinations
 
    Retour
    ------
    Each line correspond to a [main_deck_size,climax_number] combination
    Each column correspond to a probability where at least X damaged went through the damage sequence
    """
    occurrences_parsed = [is_attack(o) for o in occurrences]
    max_possible = sum(k + (1 if is_A else 0) for k, is_A in occurrences_parsed)
 
    rows = []
    for taille_deck in N_list:
        for nombre_climax in p_list:
            totals = [
                simulate_process(taille_deck, nombre_climax, occurrences_parsed, trigger_soul, trigger_blank)
                for _ in range(n_trials)
            ]
            row = {'N': taille_deck, 'p': nombre_climax}
            for x in range(0, max_possible + 1):
                row[f'X_min={x}'] = round(sum(1 for t in totals if t >= x) / n_trials, 4)
            rows.append(row)
 
    return panda.DataFrame(rows)

In [9]:
table = simulation(
        N_list=[20,25,30],
        p_list=[4,6,8],
        occurrence_list=[4,'3A',4,'3A',4,'3A'],
        trigger_soul = 15,
        trigger_blank = 35,
        n_trials=10000,
        )

table.to_csv('resultats_simulation.csv', index=False)
table

,N,p,X_min=0,X_min=1,X_min=2,X_min=3,X_min=4,X_min=5,X_min=6,X_min=7,...,X_min=15,X_min=16,X_min=17,X_min=18,X_min=19,X_min=20,X_min=21,X_min=22,X_min=23,X_min=24
0,20,4,1.0,0.2664,0.2664,0.2664,0.1253,0.0195,0.0195,0.0141,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,20,6,1.0,0.2012,0.2012,0.2012,0.0836,0.0076,0.0076,0.0050,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,20,8,1.0,0.1494,0.1494,0.1494,0.0584,0.0034,0.0034,0.0026,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,25,4,1.0,0.3017,0.3017,0.3017,0.1492,0.0290,0.0290,0.0205,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,25,6,1.0,0.2476,0.2476,0.2476,0.1139,0.0170,0.0170,0.0120,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,25,8,1.0,0.2035,0.2035,0.2035,0.0856,0.0099,0.0099,0.0060,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,30,4,1.0,0.3213,0.3213,0.3213,0.1586,0.0378,0.0378,0.0275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,30,6,1.0,0.2794,0.2794,0.2794,0.1388,0.0239,0.0239,0.0179,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,30,8,1.0,0.2331,0.2331,0.2331,0.1085,0.0170,0.0170,0.0120,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
